# 01 — Why 6 and 30? The E47 Kernel

This notebook is the lab-notebook version of `src/e47/su2_kernel.py`.

**You started with algebra and used Python as a super calculator. That is exactly right.**

This shows the same physics in two languages:
- **Symbolic:** why 6 and 30 have to appear
- **Numeric:** how we build it and measure it (carrier 125 → kernel 47)


## 1. Symbolic — 10 lines that prove the roots

For a single spin-2, $C = j(j+1) = 6$.
Couple three spin-2s. Total $J$ can be $0$ to $6$. Total Casimir eigenvalues are $C = J(J+1)$.

$K = (C - 6I)(C - 30I)$ kills exactly the $J=2$ and $J=5$ subspaces.


In [ ]:
for J in range(0, 7):
    C = J * (J + 1)
    print(f"J={J} -> C={C}")

print("\nYour roots:")
print("C=6 -> J=2")
print("C=30 -> J=5")
print("\nSo K kills J=2 and J=5. Nullspace = E47, dim = 47 inside 5^3=125")


## 2. Numeric — your super calculator, canonicalized

Build the actual 125x125 operators. This is what `su2_kernel.py` does, stripped to numpy.


In [ ]:
import numpy as np

def spin_ops(j=2):
    dim = int(2 * j + 1)
    m_vals = np.array([j - k for k in range(dim)])  # 2,1,0,-1,-2
    Jz = np.diag(m_vals)
    Jp = np.zeros((dim, dim), dtype=complex)
    for i in range(dim - 1):
        m = m_vals[i + 1]  # lower m
        # = sqrt(j(j+1)-m(m+1))
        Jp[i, i + 1] = np.sqrt(j * (j + 1) - m * (m + 1))
    Jm = Jp.T.conj()
    Jx = 0.5 * (Jp + Jm)
    Jy = -0.5j * (Jp - Jm)
    return Jx, Jy, Jz

Jx1, Jy1, Jz1 = spin_ops(2)
I5 = np.eye(5, dtype=complex)

# total J = J⊗I⊗I + I⊗J⊗I + I⊗I⊗J
Jx_tot = np.kron(np.kron(Jx1, I5), I5) + np.kron(np.kron(I5, Jx1), I5) + np.kron(np.kron(I5, I5), Jx1)
Jy_tot = np.kron(np.kron(Jy1, I5), I5) + np.kron(np.kron(I5, Jy1), I5) + np.kron(np.kron(I5, I5), Jy1)
Jz_tot = np.kron(np.kron(Jz1, I5), I5) + np.kron(np.kron(I5, Jz1), I5) + np.kron(np.kron(I5, I5), Jz1)

C = Jx_tot @ Jx_tot + Jy_tot @ Jy_tot + Jz_tot @ Jz_tot
C = 0.5 * (C + C.conj().T)  # hermitize - same as your code

I125 = np.eye(125, dtype=complex)
K = (C - 6 * I125) @ (C - 30 * I125)
K = 0.5 * (K + K.conj().T)

eigvals = np.linalg.eigvalsh(C)
print(f"Carrier dim: {C.shape[0]}")
print(f"Casimir eigenvalues present: {np.unique(np.round(eigvals).astype(int))}")

# count how close to 6 and 30 -> kernel dimension
tol = 1e-6
mask_6 = np.abs(eigvals - 6) < tol
mask_30 = np.abs(eigvals - 30) < tol
dim_J2 = np.sum(mask_6)
dim_J5 = np.sum(mask_30)
print(f"dim(C=6, J=2) = {dim_J2}")
print(f"dim(C=30, J=5) = {dim_J5}")
print(f"dim kernel K = {dim_J2 + dim_J5} <- this is your E47")
print(f"coherence fraction = {dim_J2 + dim_J5}/125 = {(dim_J2 + dim_J5) / 125}")


## 3. What this means for E47 Recursive Intelligence

- **Numeric** gives you a receipt: 125 → 47, gap 11664, max K² 186624, certificate at `artifacts/e47_validation_certificate.json`
- **Symbolic** gives you a reason: 6 and 30 aren't magic, they're $J=2$ and $J=5$

You did not waste two years. You did math first, then turned the super calculator into canonical code. That's exactly how research software should be born.

Next: add this notebook to `/notebooks/` and link it from README.
